In [7]:
import pandas as pd
import numpy as np
import os
import glob
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
import pickle
warnings.filterwarnings('ignore')

print("=== MEMULAI DATA PREPARATION ===")

# =====================================================================
# 1. LOAD DAN GABUNGKAN DATA MENTAH
# =====================================================================
print("\n1. Membaca dan menggabungkan data mentah...")

# a. Baca data cuaca
# (Sesuaikan lokasi folder jika diperlukan, misal: '../data/raw/data_cuaca_gabungan.csv')
df_weather = pd.read_csv('../data/raw/data_cuaca_gabungan.csv') 
df_weather['date'] = pd.to_datetime(df_weather['date']).dt.date
# Bulatkan koordinat untuk proses merging (karena data grid vs point)
df_weather['lat_round'] = df_weather['latitude'].round(1)
df_weather['lon_round'] = df_weather['longitude'].round(1)

# b. Baca data VIIRS
file_viirs = '../data/raw/viirs-jpss1_2018_2025_Indonesia.csv'

# Jika file gabungan belum ada, kita gabungkan ulang otomatis dari file per tahun
if not os.path.exists(file_viirs):
    print("File gabungan VIIRS tidak ditemukan. Membaca file per tahun...")
    viirs_files = glob.glob("viirs-jpss1_*_Indonesia.csv")
    df_viirs_list = []
    for f in viirs_files:
        df = pd.read_csv(f)
        df_viirs_list.append(df)
    df_viirs = pd.concat(df_viirs_list, ignore_index=True)
else:
    df_viirs = pd.read_csv(file_viirs)

df_viirs['acq_date'] = pd.to_datetime(df_viirs['acq_date']).dt.date
df_viirs['lat_round'] = df_viirs['latitude'].round(1)
df_viirs['lon_round'] = df_viirs['longitude'].round(1)

# =====================================================================
# 2. FEATURE ENGINEERING & MERGING
# =====================================================================
print("\n2. Melakukan Feature Engineering dan Merging dataset...")

# Agregasi data VIIRS: Hitung berapa banyak hotspot di titik grid tersebut pada hari tersebut
viirs_daily_grid = df_viirs.groupby(['acq_date', 'lat_round', 'lon_round']).size().reset_index(name='hotspot_count')

# Gabungkan dengan data cuaca (Left Join)
df_merged = pd.merge(
    df_weather, 
    viirs_daily_grid, 
    left_on=['date', 'lat_round', 'lon_round'], 
    right_on=['acq_date', 'lat_round', 'lon_round'], 
    how='left'
)

# Imputasi Missing Values (hari dimana tidak ada satelit yang mencatat = 0 hotspot)
df_merged['hotspot_count'] = df_merged['hotspot_count'].fillna(0)

# Buat Target Variabel Biner (1 = Ada Hotspot, 0 = Aman)
df_merged['is_hotspot'] = (df_merged['hotspot_count'] > 0).astype(int)

# Hapus kolom yang tidak diperlukan untuk pemodelan (Drop Redundant Features)
df_merged = df_merged.drop(columns=['lat_round', 'lon_round', 'acq_date', 'latitude', 'longitude', 'hotspot_count'])

# Urutkan berdasarkan waktu (Sangat krusial sebelum splitting)
df_merged = df_merged.sort_values(by='date').reset_index(drop=True)

# Definisikan X dan y
target_col = 'is_hotspot'
feature_cols = [col for col in df_merged.columns if col not in ['date', target_col]]

X = df_merged[feature_cols]
y = df_merged[target_col]

# =====================================================================
# 3. TIME-BASED DATA SPLITTING
# =====================================================================
print("\n3. Membagi data secara Kronologis (70% Train, 15% Val, 15% Test)...")

n_total = len(df_merged)
train_size = int(n_total * 0.70)
val_size = int(n_total * 0.15)

# Memotong data secara berurutan waktu (Bukan Random!)
X_train = X.iloc[:train_size].reset_index(drop=True)
y_train = y.iloc[:train_size].reset_index(drop=True)

X_val = X.iloc[train_size : train_size + val_size].reset_index(drop=True)
y_val = y.iloc[train_size : train_size + val_size].reset_index(drop=True)

X_test = X.iloc[train_size + val_size:].reset_index(drop=True)
y_test = y.iloc[train_size + val_size:].reset_index(drop=True)

print(f" - Train set : {len(X_train)} baris")
print(f" - Val set   : {len(X_val)} baris")
print(f" - Test set  : {len(X_test)} baris")

# =====================================================================
# 4. DATA TRANSFORMATION (SCALING)
# =====================================================================
print("\n4. Standarisasi Data (Fit HANYA pada Train Set)...")
scaler = StandardScaler()

# FIT dan Transform di Training
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)

# HANYA TRANSFORM di Validation dan Test (Mencegah Data Leakage)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=feature_cols)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols)

# =====================================================================
# 5. DATA BALANCING (SMOTE)
# =====================================================================
print("\n5. Menyeimbangkan kelas dengan SMOTE (HANYA pada Train Set)...")
print("Distribusi kelas sebelum SMOTE:")
print(y_train.value_counts())

# Terapkan SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print("\nDistribusi kelas setelah SMOTE:")
print(y_train_balanced.value_counts())

# =====================================================================
# 6. PENYIMPANAN DATA AKHIR
# =====================================================================
print("\n6. Menyimpan hasil persiapan data ke format CSV...")

# Buat folder jika belum ada (Opsional, sesuai struktur proyek)
if not os.path.exists('../data/processed'):
    os.makedirs('../data/processed')

X_train_balanced.to_csv('../data/processed/X_train_ready.csv', index=False)
y_train_balanced.to_csv('../data/processed/y_train_ready.csv', index=False)

X_val_scaled.to_csv('../data/processed/X_val_ready.csv', index=False)
y_val.to_csv('../data/processed/y_val_ready.csv', index=False)

X_test_scaled.to_csv('../data/processed/X_test_ready.csv', index=False)
y_test.to_csv('../data/processed/y_test_ready.csv', index=False)

# Simpan scaler agar bisa dipakai saat Inference/Deployment nanti
with open('../data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("=== DATA PREPARATION SELESAI ===")
print("File tersimpan di folder 'data/processed/'. Data siap masuk ke Tahap Modeling!")

=== MEMULAI DATA PREPARATION ===

1. Membaca dan menggabungkan data mentah...

2. Melakukan Feature Engineering dan Merging dataset...

3. Membagi data secara Kronologis (70% Train, 15% Val, 15% Test)...
 - Train set : 182882 baris
 - Val set   : 39189 baris
 - Test set  : 39190 baris

4. Standarisasi Data (Fit HANYA pada Train Set)...

5. Menyeimbangkan kelas dengan SMOTE (HANYA pada Train Set)...
Distribusi kelas sebelum SMOTE:
is_hotspot
0    182474
1       408
Name: count, dtype: int64

Distribusi kelas setelah SMOTE:
is_hotspot
0    182474
1    182474
Name: count, dtype: int64

6. Menyimpan hasil persiapan data ke format CSV...
=== DATA PREPARATION SELESAI ===
File tersimpan di folder 'data/processed/'. Data siap masuk ke Tahap Modeling!
